# 🌧️ Annual Rainfall Prediction — Year-wise Analysis

**Dataset:** NASA POWER Daily Rainfall Data (1981–2024) | Location: 12.97°N, 77.59°E

---

## 📋 Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Data Loading & Cleaning](#2-data-loading--cleaning)
3. [Annual Rainfall Aggregation & EDA](#3-annual-rainfall-aggregation--eda)
4. [Percentage Difference Analysis](#4-percentage-difference-analysis)
5. [Model 1 — Ridge Regression](#5-model-1--ridge-regression)
6. [Model 2 — Polynomial Regression (degree=5)](#6-model-2--polynomial-regression)
7. [Model 3 — Support Vector Regression (SVR)](#7-model-3--support-vector-regression)
8. [Model 4 — Random Forest Regressor](#8-model-4--random-forest-regressor)
9. [Model Comparison & Conclusion](#9-model-comparison--conclusion)

---

## 1. Setup & Imports

In [ ]:
# ── Core libraries ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11
})

RANDOM_STATE = 42
TEST_SIZE    = 0.2
PALETTE      = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']  # blue, red, green, orange

print('All libraries loaded successfully.')

---
## 2. Data Loading & Cleaning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
FILE_PATH = '/content/drive/MyDrive/POWER_Point_Daily_19810101_20240430_012d97N_077d59E_LST (1).csv'

# ── Load raw file ─────────────────────────────────────────────────────────────
raw = pd.read_csv(FILE_PATH)

# Drop the first 19 metadata rows
raw = raw.drop(raw.index[:19])

# Select relevant columns and rename
df = raw[['-BEGIN HEADER-', 'Unnamed: 1', 'Unnamed: 5']].copy()
df.columns = ['year', 'day_no', 'rainfall']

# Cast types
df['rainfall'] = df['rainfall'].astype(float)
df['year']     = df['year'].astype(int)

print(f'Shape after cleaning : {df.shape}')
print(f'Null values          :\n{df.isnull().sum()}')
df.head()

---
## 3. Annual Rainfall Aggregation & EDA

In [ ]:
# ── Aggregate daily → annual ───────────────────────────────────────────────────
annual = (
    df.groupby('year')['rainfall']
      .sum()
      .reset_index()
)
annual.columns = ['year', 'rainfall']

total_rainfall = annual['rainfall'].sum()
print(f'Total rainfall across all years : {total_rainfall:.2f} mm')
print(f'Mean annual rainfall            : {annual["rainfall"].mean():.2f} mm')
print(f'Year range                      : {annual["year"].min()} – {annual["year"].max()}')

annual.head()

In [ ]:
# ── Plot: Annual Rainfall Time Series ─────────────────────────────────────────
fig, ax = plt.subplots()
ax.plot(annual['year'], annual['rainfall'], marker='o', color=PALETTE[0], linewidth=1.8)
ax.axhline(annual['rainfall'].mean(), color='grey', linestyle='--', linewidth=1.2, label='Mean')
ax.set_title('Annual Rainfall (mm) — Year-wise', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Rainfall (mm)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Percentage Difference Analysis

**Definition:**  
$$\text{Percentage Difference} = \frac{\bar{R} - R_t}{\bar{R}} \times 100$$

where $\bar{R}$ is the long-term mean rainfall and $R_t$ is the rainfall in year $t$.

> A **positive** value means below-average rainfall; a **negative** value means above-average rainfall.

In [ ]:
mean_rainfall = annual['rainfall'].mean()
annual['pct_diff'] = ((mean_rainfall - annual['rainfall']) / mean_rainfall) * 100

print(f'Largest deficit year  : {annual.loc[annual["pct_diff"].idxmax(), "year"]}')
print(f'Largest surplus year  : {annual.loc[annual["pct_diff"].idxmin(), "year"]}')

annual.head()

In [ ]:
# ── Plot: Percentage Difference Bar Chart ──────────────────────────────────────
fig, ax = plt.subplots()
colors = [PALETTE[0] if v > 0 else PALETTE[1] for v in annual['pct_diff']]
ax.bar(annual['year'], annual['pct_diff'], color=colors, width=0.8, edgecolor='white', linewidth=0.4)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Annual Rainfall — Percentage Difference from Mean', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('% Difference')

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=PALETTE[0], label='Below average'),
    Patch(color=PALETTE[1], label='Above average')
])
plt.tight_layout()
plt.show()

---
## Helper: Evaluation Utility

A shared function to evaluate any trained model and return a clean results DataFrame.

In [ ]:
def evaluate_model(model_name, X_train, X_test, y_train, y_test, y_pred_train, y_pred_test):
    """
    Returns
    -------
    train_df  : DataFrame with train-set predictions and absolute errors
    test_df   : DataFrame with test-set  predictions and absolute errors
    mae_train : float
    mae_test  : float
    """
    train_df = pd.DataFrame({
        'year': X_train.values.flatten(),
        'actual_pct_diff': y_train.values,
        'predicted_pct_diff': y_pred_train,
        'abs_error': np.abs(y_train.values - y_pred_train),
        'split': 'train'
    })
    test_df = pd.DataFrame({
        'year': X_test.values.flatten(),
        'actual_pct_diff': y_test.values,
        'predicted_pct_diff': y_pred_test,
        'abs_error': np.abs(y_test.values - y_pred_test),
        'split': 'test'
    })

    mae_train = train_df['abs_error'].mean()
    mae_test  = test_df['abs_error'].mean()

    print(f'[{model_name}]  Train MAE: {mae_train:.4f}  |  Test MAE: {mae_test:.4f}')
    return train_df, test_df, mae_train, mae_test


def plot_actual_vs_predicted(annual, test_df, model_name):
    """Overlay the full actual series with the test-set predictions."""
    fig, ax = plt.subplots()
    ax.plot(annual['year'], annual['pct_diff'],
            marker='o', linestyle='-', color=PALETTE[0], label='Actual', linewidth=1.5)
    ax.scatter(test_df['year'], test_df['predicted_pct_diff'],
               marker='x', color=PALETTE[1], s=80, label='Predicted (test)', zorder=5)
    ax.set_title(f'Actual vs Predicted — {model_name}', fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('% Difference from Mean Rainfall')
    ax.legend()
    plt.tight_layout()
    plt.show()


# ── Shared train/test split used by all models ────────────────────────────────
X = annual[['year']]
y = annual['pct_diff']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f'Train size: {len(X_train)}  |  Test size: {len(X_test)}')

---
## 5. Model 1 — Ridge Regression

A regularised linear model (L2 penalty) used as a strong baseline.

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────────
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

ridge_train_df, ridge_test_df, ridge_mae_train, ridge_mae_test = evaluate_model(
    'Ridge Regression',
    X_train, X_test, y_train, y_test,
    ridge.predict(X_train),
    ridge.predict(X_test)
)
ridge_test_df

In [ ]:
plot_actual_vs_predicted(annual, ridge_test_df, 'Ridge Regression')

---
## 6. Model 2 — Polynomial Regression

Degree-5 polynomial features fitted with OLS.  
Captures non-linear trends that a linear model misses.

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
poly = PolynomialFeatures(degree=5)
X_train_poly = poly.fit_transform(X_train)
X_test_poly  = poly.transform(X_test)

# ── Train ─────────────────────────────────────────────────────────────────────
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

poly_train_df, poly_test_df, poly_mae_train, poly_mae_test = evaluate_model(
    'Polynomial Regression (deg=5)',
    X_train, X_test, y_train, y_test,
    poly_model.predict(X_train_poly),
    poly_model.predict(X_test_poly)
)
poly_test_df

In [ ]:
plot_actual_vs_predicted(annual, poly_test_df, 'Polynomial Regression (deg=5)')

---
## 7. Model 3 — Support Vector Regression (SVR)

Linear kernel SVR; robust to outliers via the ε-insensitive loss.

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
svr = SVR(kernel='linear')
svr.fit(X_train, y_train)

svr_train_df, svr_test_df, svr_mae_train, svr_mae_test = evaluate_model(
    'SVR (linear)',
    X_train, X_test, y_train, y_test,
    svr.predict(X_train),
    svr.predict(X_test)
)
svr_test_df

In [ ]:
plot_actual_vs_predicted(annual, svr_test_df, 'SVR (linear kernel)')

---
## 8. Model 4 — Random Forest Regressor

Ensemble of decision trees; naturally handles non-linearities and tends to overfit on small datasets — watch the train/test gap.

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
rf = RandomForestRegressor(random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

rf_train_df, rf_test_df, rf_mae_train, rf_mae_test = evaluate_model(
    'Random Forest',
    X_train, X_test, y_train, y_test,
    rf.predict(X_train),
    rf.predict(X_test)
)
rf_test_df

In [ ]:
plot_actual_vs_predicted(annual, rf_test_df, 'Random Forest')

---
## 9. Model Comparison & Conclusion

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Model': ['Ridge Regression', 'Polynomial Regression (deg=5)', 'SVR (linear)', 'Random Forest'],
    'Train MAE': [ridge_mae_train, poly_mae_train, svr_mae_train, rf_mae_train],
    'Test MAE':  [ridge_mae_test,  poly_mae_test,  svr_mae_test,  rf_mae_test]
}).sort_values('Test MAE').reset_index(drop=True)

comparison

In [ ]:
# ── Bar Chart: Test MAE Comparison ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(comparison['Model'], comparison['Test MAE'],
               color=PALETTE[:len(comparison)], edgecolor='white')
ax.bar_label(bars, fmt='%.3f', padding=4)
ax.set_xlabel('Mean Absolute Error (Test Set)')
ax.set_title('Model Comparison — Test MAE (lower is better)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Overlay: All Models on Actual Series ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(annual['year'], annual['pct_diff'],
        color='black', linewidth=2, marker='o', markersize=4, label='Actual')

for test_df, label, color in [
    (ridge_test_df, 'Ridge',      PALETTE[0]),
    (poly_test_df,  'Polynomial', PALETTE[1]),
    (svr_test_df,   'SVR',        PALETTE[2]),
    (rf_test_df,    'RandomForest', PALETTE[3]),
]:
    sorted_df = test_df.sort_values('year')
    ax.scatter(sorted_df['year'], sorted_df['predicted_pct_diff'],
               color=color, s=80, zorder=5, label=label)

ax.set_title('All Models — Predicted vs Actual (Test Years)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('% Difference from Mean Rainfall')
ax.legend()
plt.tight_layout()
plt.show()

---

### 📝 Findings

| Aspect | Observation |
|---|---|
| Best Test MAE | See `comparison` table above |
| Random Forest overfitting | Train MAE ≈ 0; wide train/test gap — expected on a small dataset |
| Polynomial Regression | Flexible curve, but may overfit at degree 5; consider cross-validation |
| Ridge / SVR | More stable generalisation on a dataset of ~43 annual observations |

### ✅ Recommendations
- For **generalisability** on this small time series (~43 rows), prefer Ridge or SVR.
- Consider **Leave-One-Out CV** instead of a single train/test split.
- Adding climate indices (ENSO, IOD) as features could substantially improve all models.

---
*Notebook generated for Annual Rainfall Prediction — Year-wise Analysis.*